# Tutorial 3: Inference

Once you have a trained model, you may use it to run inference on new data with `elemenet_inference`. This loads the run's `config.yaml` and best model checkpoint, rebuilds the same preprocessing, predicts, and (if ground-truth labels are present) reports summary statistics.

## CLI usage

Point `--model_path` at the run directory (the folder containing `config.yaml`), and `--new_data_path` at a CSV of new molecules. Match `--graph_format`, `--mol_column`, and (optionally) `--target_column` to how the original model was trained.

In [ ]:
!elemenet_inference \
    --model_path model \
    --new_data_path datasets/QM9-test.csv \
    --mol_column smiles \
    --graph_format smiles \
    --save_dir inference_results \
    --target_column u0_atom #optional

### Arguments

| Flag | Meaning |
|------|---------|
| `--model_path` | Run directory containing `config.yaml` and `checkpoints/`. |
| `--new_data_path` | CSV with the molecules to predict on. |
| `--mol_column` | Column with the molecule representation (must match training). |
| `--graph_format` | Molecular graph type (must match training). |
| `--target_column` | Target column name(s) (optional). Comma-separated for multi-target. |
| `--batch_size` | Inference batch size. |
| `--save_dir` | Output directory (default `inference_results`). |
| `--device` | `cuda` (default) or `cpu`. |

## Outputs

Results are stored in `--save_dir`:

```
inference_results/
├── stats/
│   ├── test_predictions.csv   # per-molecule predictions (+ ground truth if available)
│   └── test_metrics.json      # aggregate metrics (when labels are present)
└── data/                      # cached processed graphs + scaled targets
```

To load and inspect the predictions:

In [ ]:
import json
import pandas as pd

preds = pd.read_csv("inference_results/stats/test_predictions.csv")
display(preds.head())

try:
    with open("inference_results/stats/test_metrics.json") as f:
        print(json.dumps(json.load(f), indent=2))
except FileNotFoundError:
    print("No metrics file — inference ran on unlabeled data.")

## SLURM example

```bash
#!/bin/bash
#SBATCH --job-name=elemenet_inference
#SBATCH -o prediction_test.out
#SBATCH --gres=gpu:1
#SBATCH --mem=64G
#SBATCH -c 4

conda activate ElemeNet

elemenet_inference --model_path model --new_data_path datasets/QM9-test.csv \
    --mol_column mol2 --graph_format mol2 --target_column mu
```